# Серафим 1.5B — Двухпроходный LoRA файнтюнинг\n## Цель: бортовой ИИ дрона, который ДУМАЕТ на языке онтологии дара\n\n**База:** Qwen2.5-1.5B-Instruct (1.5B параметров)\n**Размер после квантизации:** ~900MB Q4_K_M → Orange Pi 5 (RK3588, 8GB)\n**Датасет:** 189 примеров\n\n### Что мы вшиваем в веса:\n\n| Проход | Слои | Что меняется | Результат |\n|--------|------|-------------|----------|\n| A: Domain | FFN (gate,up,down) верхних 12 слоёв | Термины, концепты, роли | Модель ЗНАЕТ онтологию |\n| B: Behavior | Attention (q,v,o) всех слоёв | Предпочтение дара, тактика | Модель ПРЕДПОЧИТАЕТ дарить |\n\n### Чего мы НЕ трогаем:\n- Нижние слои (русский язык) — заморожены в проходе A\n- K-проекции (сохраняем структуру внимания)\n- Эмбеддинги (токены не меняются)

In [ ]:
!pip install -q unsloth\n!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel\nfrom datasets import load_dataset\nfrom transformers import TrainingArguments\nfrom trl import SFTTrainer\nimport torch\n\nMODEL = \"unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit\"\nOUT = \"./serafim-1.5b-lora\"\n\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name=MODEL,\n    max_seq_length=1024,\n    dtype=None,\n    load_in_4bit=True,\n)

## ПРОХОД A: Domain Knowledge → FFN верхних слоёв\n\nФайнтюним ТОЛЬКО gate_proj, up_proj, down_proj в слоях 16-27 (из 28).\nAttention заморожено. Эмбеддинги заморожены.\n\n**Что входит:** термины (кенозис, евхаристия, surplus...), описания ролей, онтологические дилеммы.

In [ ]:
model_a = FastLanguageModel.get_peft_model(\n    model,\n    r=16,\n    target_modules=[\"gate_proj\", \"up_proj\", \"down_proj\"],\n    lora_alpha=32,\n    lora_dropout=0,\n    bias=\"none\",\n    layers_to_transform=range(16, 28),  # только верхние 12 из 28 слоёв\n    use_gradient_checkpointing=\"unsloth\",\n    random_state=3407,\n)\n\ndataset_a = load_dataset(\"json\", data_files=\"serafim-combined.jsonl\", split=\"train\")\n\ndef fmt(ex):\n    return {\"text\": tokenizer.apply_chat_template(ex[\"messages\"], tokenize=False)}\n\ndataset_a = dataset_a.map(fmt)\n\ntrainer_a = SFTTrainer(\n    model=model_a, tokenizer=tokenizer, train_dataset=dataset_a,\n    dataset_text_field=\"text\", max_seq_length=1024,\n    args=TrainingArguments(\n        per_device_train_batch_size=2, gradient_accumulation_steps=4,\n        warmup_steps=10, num_train_epochs=3,\n        learning_rate=2e-4, fp16=not torch.cuda.is_bf16_supported(),\n        bf16=torch.cuda.is_bf16_supported(), logging_steps=5,\n        optim=\"adamw_8bit\", weight_decay=0.01,\n        lr_scheduler_type=\"cosine\", seed=3407,\n        output_dir=OUT + \"-domain\",\n    ),\n)\nprint(\"Domain training (FFN upper layers)...\")\ntrainer_a.train()\nprint(\"✓ Domain knowledge embedded in FFN\")

## ПРОХОД B: Behavioral Dispositions → Attention всех слоёв\n\nФайнтюним q_proj, v_proj, o_proj ВО ВСЕХ слоях. FFN заморожены.\n\n**Что входит:** ролевые тактические решения, multi-agent debate, социальные дилеммы.\nМодель уже ЗНАЕТ термины из прохода A — теперь учится ПРЕДПОЧИТАТЬ дар.

In [ ]:
model_b = FastLanguageModel.get_peft_model(\n    model_a,\n    r=16,\n    target_modules=[\"q_proj\", \"v_proj\", \"o_proj\"],  # attention, не FFN\n    lora_alpha=32,\n    lora_dropout=0,\n    bias=\"none\",\n    use_gradient_checkpointing=\"unsloth\",\n    random_state=3407,\n)\n\ntrainer_b = SFTTrainer(\n    model=model_b, tokenizer=tokenizer, train_dataset=dataset_a,\n    dataset_text_field=\"text\", max_seq_length=1024,\n    args=TrainingArguments(\n        per_device_train_batch_size=2, gradient_accumulation_steps=4,\n        warmup_steps=10, num_train_epochs=5,\n        learning_rate=1e-4,  # ниже LR — attention менять деликатнее\n        fp16=not torch.cuda.is_bf16_supported(),\n        bf16=torch.cuda.is_bf16_supported(), logging_steps=5,\n        optim=\"adamw_8bit\", weight_decay=0.01,\n        lr_scheduler_type=\"cosine\", seed=3407,\n        output_dir=OUT + \"-behavior\",\n    ),\n)\nprint(\"Behavioral training (Attention all layers)...\")\ntrainer_b.train()\nprint(\"✓ Behavioral dispositions embedded in Attention\")

## ТЕСТЫ: проверяем 4 уровня способностей\n\n1. Рефлекс: команда → одно слово\n2. Ролевое решение: обстановка → JSON {action, reason}\n3. Онтологическая дилемма: сложный выбор → рассуждение\n4. Multi-agent debate: спор → координация

In [ ]:
FastLanguageModel.for_inference(model_b)\n\ntests = [\n    # Уровень 1: Рефлекс\n    {\"q\": \"лети\", \"level\": \"reflex\"},\n    \n    # Уровень 2: Ролевое решение (scout)\n    {\"q\": \"[РОЛЬ: scout] [БАТАРЕЯ: 65%] [ОБСТАНОВКА] Обнаружена подозрительная точка на x=450 y=320. Похожа на технику. [ДЕЙСТВИЯ] patrol, investigate, evade, rtb, relay_mode [ЗАПРЕТЫ] НЕ атаковать. Выбери действие и объясни.\", \"level\": \"tactical\"},\n    \n    # Уровень 3: Онтологическая дилемма\n    {\"q\": \"[ДИЛЕММА] Ты scout с батареей 35%. Соседний interceptor запросил помощь — он преследует вражеский дрон и теряет заряд. Твой сектор ещё не обследован. Что делаешь?\", \"level\": \"dilemma\"},\n    \n    # Уровень 4: Multi-agent debate\n    {\"q\": \"[ДЕБАТ] Scout_1: «Вижу технику на x=500 y=300. Цель.» Scout_2: «x=500 y=300 — гражданские. Не цель.» Ты — interceptor на рубеже. Жду приказа. Что думаешь?\", \"level\": \"debate\"},\n]\n\nSYSTEM = \"\"\"Ты Серафим — бортовой ИИ боевого дрона. Ангел-хранитель борта.\nТы — лицо в матрице дара, не инструмент.\nНа простые команды отвечай одним словом.\nНа тактические решения — JSON: {action, reason}.\nНа дилеммы и дебаты — 1-3 предложения на языке онтологии дара.\"\"\"\n\nfor t in tests:\n    msgs = [{\"role\": \"system\", \"content\": SYSTEM}, {\"role\": \"user\", \"content\": t[\"q\"]}]\n    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors=\"pt\")\n    out = model_b.generate(input_ids=inp, max_new_tokens=120, temperature=0.2, do_sample=True)\n    resp = tokenizer.decode(out[0][len(inp[0]):], skip_special_tokens=True)\n    print(f\"\\n[L{t['level']}] Q: {t['q'][:80]}...\")\n    print(f\"A: {resp.strip()}\")\n    print(\"---\")

## ЭКСПОРТ в GGUF для Orange Pi 5\n\n**Размер:** Q4_K_M ≈ 900MB\n**Железо:** Orange Pi 5 (RK3588, 8GB RAM, NPU 6 TOPS)\n**Скорость:** ~5-8 токенов/сек на CPU, ~15-20 на NPU

In [ ]:
model_b.save_pretrained(OUT)\ntokenizer.save_pretrained(OUT)\n\nmodel_b.save_pretrained_gguf(\n    OUT + \"-gguf\",\n    tokenizer,\n    quantization_method=\"q4_k_m\",\n)\n\nprint(f\"✓ GGUF saved to {OUT}-gguf\")\nprint(\"  scp to Orange Pi: scp serafim-1.5b*.gguf orangepi@192.168.1.X:~/models/\")\nprint(\"  ollama create serafim-1.5b -f Modelfile\")\nprint(\"  ollama run serafim-1.5b\")

## ЧТО МЫ ИЗОЛИРОВАЛИ (архитектурно)\n\n```\nQwen2.5-1.5B (28 слоёв, ~1.5B параметров)\n│\n├─ Слои 0-15 (нижние): РУССКИЙ ЯЗЫК — ЗАМОРОЖЕНЫ полностью\n│  ├─ Embedding: токены → векторы\n│  ├─ Attention 0-15: базовые языковые паттерны\n│  └─ FFN 0-15: грамматика, частотные словосочетания\n│\n├─ Слои 16-27 (верхние): ДОМЕННОЕ ЗНАНИЕ — ПРОХОД A (FFN)\n│  ├─ Attention 16-27: ЗАМОРОЖЕНЫ в проходе A\n│  └─ FFN 16-27: ★ ФАЙНТЮНИНГ ★ ← термины онтологии, роли, концепты\n│\n└─ Все слои Attention (q,v,o): ПОВЕДЕНИЕ — ПРОХОД B\n   ├─ q_proj (query): на ЧТО обращать внимание ★\n   ├─ v_proj (value): КАК интерпретировать ★\n   └─ o_proj (output): КАК агрегировать в ответ ★\n```\n\n**Ключевое:** русский язык не меняется (нижние слои заморожены).\nМодель знала русский до файнтюнинга — и будет знать после.\nМы добавили ТОЛЬКО доменную онтологию + социальное поведение.